In [1]:
# ============================================================
# CELL 1: Imports + Konfiguration
# ============================================================
import os
import shutil
import torch
import pandas as pd
from tqdm import tqdm
from PIL import Image, UnidentifiedImageError
from transformers import CLIPProcessor, CLIPModel
import open_clip
import mobileclip

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Gerät: {device}")

categories = ["individual", "community_ecosystem", "landscape"]
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# --- Pfade ---
IMAGE_DIR  = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Individual_CoEco_Landscape\30062026_random_sample_images_Individual_CoEco_Landscape"
OUTPUT_DIR = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Individual_CoEco_Landscape\model_predictions"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MOBILECLIP_WEIGHTS = r"C:\Users\Lukas\masterthesis\thesis_2026\data\weights\mobileclip_weights\mobileclip_s1.pt"

MODEL_CONFIGS = [
    {"name": "CLIP-ViT-B32",   "type": "hf",         "model_id": "openai/clip-vit-base-patch32",  "batch_size": 64},
    {"name": "CLIP-ViT-L14",   "type": "hf",         "model_id": "openai/clip-vit-large-patch14", "batch_size": 32},
    {"name": "EVA01-CLIP-g14", "type": "open_clip",  "model_id": "EVA01-g-14", "pretrained": "laion400m_s11b_b41k", "batch_size": 16},
    {"name": "MobileCLIP-S1",  "type": "mobileclip", "model_id": "mobileclip_s1", "pretrained": MOBILECLIP_WEIGHTS, "batch_size": 32},
]


# ============================================================
# CELL 2: Optimierte Mixed-Prompts (MobileCLIP-S1, Dev-Evaluation)
# ============================================================
# Basierend auf der Modell-Evaluation:
# MobileCLIP-S1 + Mixed = bestes Modell/Prompt-Set (Macro-F1 = 0.787)
# Alle drei Klassen: Mixed-Prompt am besten
mixed_prompts = {
    "individual":          "a close-up photo with a single animal, plant, or object as the "
                           "clear main subject",
    "community_ecosystem": "a photo dominated by multiple organisms, people, buildings, "
                           "vehicles, or man-made structures",
    "landscape":           "a wide natural landscape photo with visible horizon or open "
                           "terrain, no dominant single subject or man-made structures"
}

category_texts = [mixed_prompts[cat] for cat in categories]


# ============================================================
# CELL 3: Modell-Lade- und Encode-Funktionen
# ============================================================
def load_model(config):
    if config["type"] == "hf":
        processor = CLIPProcessor.from_pretrained(config["model_id"])
        model = CLIPModel.from_pretrained(config["model_id"]).to(device)
        model.eval()
        return {"type": "hf", "model": model, "processor": processor}
    elif config["type"] == "open_clip":
        model, _, preprocess = open_clip.create_model_and_transforms(
            config["model_id"], pretrained=config["pretrained"]
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer(config["model_id"])
        return {"type": "open_clip", "model": model, "preprocess": preprocess, "tokenizer": tokenizer}
    elif config["type"] == "mobileclip":
        model, _, preprocess = mobileclip.create_model_and_transforms(
            config["model_id"], pretrained=config["pretrained"], device=device
        )
        model.eval()
        tokenizer = mobileclip.get_tokenizer(config["model_id"])
        return {"type": "mobileclip", "model": model, "preprocess": preprocess, "tokenizer": tokenizer}

def encode_text(mb, texts):
    if mb["type"] == "hf":
        inputs = mb["processor"](
            text=texts, return_tensors="pt", padding=True, truncation=True
        ).to(device)
        with torch.no_grad():
            out = mb["model"].get_text_features(**inputs)
        feats = out if isinstance(out, torch.Tensor) else out.pooler_output
    elif mb["type"] in ("open_clip", "mobileclip"):
        tokens = mb["tokenizer"](texts).to(device)
        with torch.no_grad():
            feats = mb["model"].encode_text(tokens)
    return feats / feats.norm(dim=-1, keepdim=True)

def encode_images(mb, imgs):
    if mb["type"] == "hf":
        inputs = mb["processor"](
            images=list(imgs), return_tensors="pt", padding=True
        ).to(device)
        with torch.no_grad():
            out = mb["model"].get_image_features(**inputs)
        feats = out if isinstance(out, torch.Tensor) else out.pooler_output
    elif mb["type"] in ("open_clip", "mobileclip"):
        tensors = torch.stack([mb["preprocess"](img) for img in imgs]).to(device)
        with torch.no_grad():
            feats = mb["model"].encode_image(tensors)
    return feats / feats.norm(dim=-1, keepdim=True)

def safe_open(path):
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError) as e:
        print(f"  ⚠ Skipped ({os.path.basename(path)}): {e}")
        return None


# ============================================================
# CELL 4: Alle Bilder sammeln
# ============================================================
image_paths = []
for fname in sorted(os.listdir(IMAGE_DIR)):
    ext = os.path.splitext(fname)[1].lower()
    if ext in VALID_EXTENSIONS:
        image_paths.append(os.path.join(IMAGE_DIR, fname))

print(f"Bilder gefunden: {len(image_paths)}")


# ============================================================
# CELL 5: Klassifikation mit allen 4 Modellen
# ============================================================
all_results = {}

for config in MODEL_CONFIGS:
    model_name = config["name"]
    batch_size = config["batch_size"]
    print(f"\n{'='*60}\nModell: {model_name}\n{'='*60}")

    model_bundle  = load_model(config)
    text_features = encode_text(model_bundle, category_texts)

    results, confidences, valid_paths = [], [], []

    for i in tqdm(range(0, len(image_paths), batch_size), desc="Klassifiziere"):
        batch_paths = image_paths[i : i + batch_size]
        loaded = [(p, safe_open(p)) for p in batch_paths]
        loaded = [(p, img) for p, img in loaded if img is not None]
        if not loaded:
            continue

        paths_ok, imgs_ok = zip(*loaded)
        image_features = encode_images(model_bundle, imgs_ok)
        similarities   = image_features @ text_features.T
        batch_preds    = similarities.argmax(dim=1).cpu().numpy()
        batch_conf     = similarities.max(dim=1).values.cpu().float().numpy()

        results.extend([categories[idx] for idx in batch_preds])
        confidences.extend(batch_conf.tolist())
        valid_paths.extend(paths_ok)

    # --- CSV speichern ---
    df = pd.DataFrame({
        "file_name":  [os.path.basename(p) for p in valid_paths],
        "file_path":  valid_paths,
        "pred_label": results,
        "confidence": confidences,
    })

    csv_path = os.path.join(OUTPUT_DIR, f"predictions_{model_name}.csv")
    df.to_csv(csv_path, index=False)
    print(f"✅ CSV gespeichert: {csv_path}")
    print(df["pred_label"].value_counts())

    all_results[model_name] = df

    # --- Bilder in Unterordner sortieren ---
    model_sort_dir = os.path.join(OUTPUT_DIR, model_name)
    for cat in categories:
        os.makedirs(os.path.join(model_sort_dir, cat), exist_ok=True)

    for path, pred in zip(valid_paths, results):
        dest = os.path.join(model_sort_dir, pred, os.path.basename(path))
        if not os.path.exists(dest):
            shutil.copy(path, dest)

    print(f"Bilder sortiert in: {model_sort_dir}")

    del model_bundle
    torch.cuda.empty_cache()


# ============================================================
# CELL 6: Kombinierte Übersichtstabelle aller Modelle
# ============================================================
df_combined = pd.DataFrame({
    "file_name": [os.path.basename(p) for p in image_paths]
})

for model_name, df in all_results.items():
    df_merged = df[["file_name", "pred_label", "confidence"]].rename(columns={
        "pred_label": f"pred_{model_name}",
        "confidence": f"conf_{model_name}",
    })
    df_combined = df_combined.merge(df_merged, on="file_name", how="left")

combined_csv = os.path.join(OUTPUT_DIR, "all_models_predictions_combined.csv")
df_combined.to_csv(combined_csv, index=False)
print(f"\n✅ Kombinierte Übersicht gespeichert: {combined_csv}")
print(df_combined.head())

C:\Users\Lukas\miniconda3\envs\kilimanjaro\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Lukas\miniconda3\envs\kilimanjaro\lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Gerät: cuda
Bilder gefunden: 297

Modell: CLIP-ViT-B32


Klassifiziere: 100%|█████████████████████████████████████████████████████████████████████| 5/5 [00:29<00:00,  5.85s/it]


✅ CSV gespeichert: C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Individual_CoEco_Landscape\model_predictions\predictions_CLIP-ViT-B32.csv
pred_label
landscape              231
individual              36
community_ecosystem     30
Name: count, dtype: int64
Bilder sortiert in: C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Individual_CoEco_Landscape\model_predictions\CLIP-ViT-B32

Modell: CLIP-ViT-L14


Klassifiziere: 100%|███████████████████████████████████████████████████████████████████| 10/10 [01:00<00:00,  6.05s/it]


✅ CSV gespeichert: C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Individual_CoEco_Landscape\model_predictions\predictions_CLIP-ViT-L14.csv
pred_label
landscape              208
community_ecosystem     61
individual              28
Name: count, dtype: int64
Bilder sortiert in: C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Individual_CoEco_Landscape\model_predictions\CLIP-ViT-L14

Modell: EVA01-CLIP-g14


Klassifiziere: 100%|███████████████████████████████████████████████████████████████████| 19/19 [24:06<00:00, 76.13s/it]


✅ CSV gespeichert: C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Individual_CoEco_Landscape\model_predictions\predictions_EVA01-CLIP-g14.csv
pred_label
landscape              143
community_ecosystem     99
individual              55
Name: count, dtype: int64
Bilder sortiert in: C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Individual_CoEco_Landscape\model_predictions\EVA01-CLIP-g14

Modell: MobileCLIP-S1


Klassifiziere: 100%|███████████████████████████████████████████████████████████████████| 10/10 [00:17<00:00,  1.71s/it]


✅ CSV gespeichert: C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Individual_CoEco_Landscape\model_predictions\predictions_MobileCLIP-S1.csv
pred_label
landscape              211
community_ecosystem     53
individual              33
Name: count, dtype: int64
Bilder sortiert in: C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Individual_CoEco_Landscape\model_predictions\MobileCLIP-S1

✅ Kombinierte Übersicht gespeichert: C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Individual_CoEco_Landscape\model_predictions\all_models_predictions_combined.csv
         file_name pred_CLIP-ViT-B32  conf_CLIP-ViT-B32    pred_CLIP-ViT-L14  \
0  image_10016.jpg         landscape           0.252337            landscape   
1  image_10030.jpg         landscape           0.270158            landscape   
2  image_10038.jpg         landscape           0.289032            landscape   
3  image_10062.jpg         landscape           0.259290  community_ecosystem   
4  image_10064.jpg    